# Modelo de refletividade 2D sintético

Resumo da metodologia implementada em [`processamento/refletividade_2d.py`](../processamento/refletividade_2d.py), com as funções auxiliares em [`processamento/utils.py`](../processamento/utils.py).

O objetivo é gerar uma seção sísmica sintética 2D (somente reflexões) seguindo três etapas:

1. preencher a seção com refletores controlados por duas tendências senoidais;
2. convolver cada traço com uma mesma wavelet Ricker;
3. adicionar ruído gaussiano com SNR aleatório.

## Parâmetros

| Parâmetro | Valor | Descrição |
|---|---|---|
| `ns` | 256 | amostras por traço |
| `ntr` | 256 | número de traços |
| `nrefl` | 10 | número de refletores |
| `dt` | 0,006 s | intervalo de amostragem |
| `t1`, `t2` | 0,3 – 0,4 | faixa relativa (fração de `ns`) da posição média do topo |
| `b1`, `b2` | 0,9 – 0,99 | faixa relativa (fração de `ns`) da posição média da base |

Todos os sorteios usam um único gerador `rng = np.random.default_rng()`; $\mathcal{U}(a,b)$ denota distribuição uniforme.

## 1. Tendências senoidais (topo e base)

Duas curvas, topo e base, definem o "envelope" onde os refletores são posicionados. Cada uma é uma senoide em função do traço $x$ (tempos em amostras):

$$
t(x) = \bar{t} + A \,\sin\!\left(\frac{2\pi x}{\lambda} + \varphi\right), \qquad x = 0, \dots, n_{tr}-1
$$

com

- $A \sim \mathcal{U}(0,\ 0{,}1\,n_s)$: amplitude da ondulação;
- $\lambda \sim \mathcal{U}(0{,}5\,n_{tr},\ 3\,n_{tr})$: comprimento de onda (ondulações suaves, no máximo ~2 ciclos na seção);
- $\varphi \sim \mathcal{U}(0,\ 2\pi)$: fase;
- $\bar{t}$: tempo médio da curva, isto é, o nível em torno do qual a senoide oscila.

Como $\sin(\cdot) \in [-1, 1]$, a curva fica sempre na faixa $\bar{t} - A \le t(x) \le \bar{t} + A$: $\bar{t}$ define a **profundidade** (em tempo) da tendência e $A$ o quanto ela **ondula** em torno dela.

O tempo médio é sorteado como fração de $n_s$:

$$
\bar{t}_{topo} \sim \mathcal{U}(t_1, t_2)\,n_s, \qquad \bar{t}_{base} \sim \mathcal{U}(b_1, b_2)\,n_s
$$

Com $n_s = 256$, o topo fica centrado entre as amostras ~77 e ~102 (30–40% da seção) e a base entre ~230 e ~253, perto do fundo. Como $A \le 0{,}1\,n_s \approx 25$ amostras, cada curva se afasta no máximo ~25 amostras do seu $\bar{t}$; por isso `t1, t2` e `b1, b2` controlam, na prática, onde ficam o topo e a base.

## 2. Posicionamento dos refletores

Cada refletor $k$ recebe uma **posição relativa** $u_k$ e uma **amplitude** $a_k$:

$$
u_k \sim \mathcal{U}(-0{,}1,\ 1{,}1) \ \text{(ordenados)}, \qquad a_k \sim \mathcal{U}(-1,\ 1)
$$

O tempo do refletor em cada traço é uma interpolação linear entre topo e base:

$$
t_k(x) = t_{topo}(x) + u_k\,\big[t_{base}(x) - t_{topo}(x)\big]
$$

- $u_k = 0$: refletor sobre o topo; $u_k = 1$: sobre a base;
- $u_k < 0$ ou $u_k > 1$: um pouco acima do topo ou abaixo da base.

Assim, os refletores acompanham a geometria das duas tendências, ficando mais "espremidos" onde topo e base se aproximam.

### Distribuição sub-amostral

Como $t_k(x)$ geralmente não é inteiro, a amplitude é dividida entre as duas amostras vizinhas (interpolação linear):

$$
i_0 = \lfloor t_k(x) \rfloor, \qquad w = t_k(x) - i_0
$$

$$
r(i_0, x) \mathrel{+}= a_k\,(1 - w), \qquad r(i_0 + 1, x) \mathrel{+}= a_k\,w
$$

A amostra mais próxima de $t_k$ recebe a maior parte da amplitude. Índices fora de $[0, n_s)$ são descartados, e refletores que caem na mesma amostra se somam.

## 3. Convolução com a wavelet Ricker

A wavelet Ricker com frequência de pico $f_p$ é

$$
w(t) = \left(1 - 2\pi^2 f_p^2 t^2\right)\, e^{-\pi^2 f_p^2 t^2}, \qquad t \in \left[-\tfrac{n_s\,\Delta t}{2},\ \tfrac{n_s\,\Delta t}{2}\right)
$$

com $f_p \sim \mathcal{U}(10,\ 40)$ Hz e as mesmas $n_s = 256$ amostras com $\Delta t = 6$ ms da seção.

Pelo modelo convolucional, cada traço da seção é a convolução da refletividade com a **mesma** wavelet:

$$
s(t, x) = r(t, x) * w(t)
$$

A convolução usa `np.convolve(..., mode="same")`, mantendo $n_s$ amostras por traço (`convolve_secao`).

## 4. Ruído gaussiano

O ruído é aditivo, gaussiano e de média zero, com variância calculada para atingir uma razão sinal-ruído (em dB) sorteada $\text{SNR} \sim \mathcal{U}(5,\ 20)$ dB (`adiciona_ruido_snr`):

$$
P_s = \frac{1}{n_s\,n_{tr}} \sum_{t,x} s(t,x)^2, \qquad
P_n = \frac{P_s}{10^{\,\text{SNR}/10}}
$$

$$
\tilde{s}(t,x) = s(t,x) + n(t,x), \qquad n \sim \mathcal{N}\!\left(0,\ P_n\right)
$$

Quanto menor o SNR, mais ruidosa a seção.